# Module 1: Define Architecture

## Core Objective
To systematically analyze a cohort of genomes, identify the conserved structural backbone, and produce a single, comprehensive blueprint (`architecture_definition.json`) for all downstream analyses.

## Key Workflow & Features

1.  **Data Loading & Indexing**:
    *   Parses the Panaroo pangenome matrix to stratify genes into `Strict Core`, `Soft Core`, and `Accessory`.
    *   Processes all GFF files to build databases of gene order and physical coordinates for each genome.

2.  **Rigorous Quality Control**:
    *   Performs an assembly integrity check by counting contigs directly from the source FASTA files. Genomes that are too fragmented (e.g., >1 contig) are excluded from structural analysis.
    *   Checks for a consistent starting gene across the cohort to determine if the dataset is uniformly oriented.

3.  **Anchor Module Discovery**:
    *   Algorithmically discovers "Anchor Modules"—contiguous blocks of strict core genes that are conserved across a high fraction of the genomes (`--coverage-threshold`).
    *   The algorithm is tolerant to a small number of intervening accessory genes (`--tolerance`).

4.  **Anchor Characterization & Annotation**:
    *   Quantifies the intrinsic stability of each anchor using two key metrics:
        *   **ASI (Anchor Sequence Identity)**: Measures sequence-level conservation.
        *   **AOI (Anchor Order Index)**: Measures the rigidity of the internal gene order.
    *   Assigns a dominant biological function to each anchor based on the annotations of its member genes.

5.  **Macro-Architectural Analysis**:
    *   Determines the physical order (permutation) of anchor modules for each high-quality genome.
    *   Identifies the "Major Architecture" (the most common permutation) and calculates the global structural entropy to quantify the overall architectural rigidity of the cohort.

In [ ]:
# /GenoSynax_test/genosyntax/define_architecture/__init__.py

import os
import json
import pandas as pd

from genosyntax.define_architecture.data_loader import load_panaroo_data, load_gff_data, load_protein_sequences, load_genome_sequences
from genosyntax.define_architecture.quality_control import run_qc
from genosyntax.define_architecture.anchor_discovery import discover_anchors, characterize_anchors
from genosyntax.define_architecture.functional_annotator import annotate_anchor_functions
from genosyntax.define_architecture.macro_analysis import analyze_macro_architecture
from genosyntax.utils.file_ops import save_json

def run(args):
    """
    Entrypoint function for the 'define_architecture' module.
    MODIFIED: Now handles the case where no anchors are discovered, providing a
    clear warning and generating a valid but empty architecture file to prevent
    downstream crashes.
    """
    # --- Setup Directories ---
    output_dir = args.output_dir
    intermediate_dir = os.path.join(output_dir, "intermediate")
    reports_dir = os.path.join(output_dir, "reports")
    os.makedirs(intermediate_dir, exist_ok=True)
    os.makedirs(reports_dir, exist_ok=True)

    if args.dry_run:
        # ... (dry_run logic is correct, no changes)
        return True

    print("====== GenoSyntax: Running Module 'define_architecture' ======")
    
    # --- Step 1 & 2 : Data Loading & QC  ---
    genome_ids, stratified_genes, locus_to_group_map = load_panaroo_data(
        args.panaroo_file,
        strict_core_threshold=args.strict_core_threshold,
        soft_core_threshold=args.soft_core_threshold
    )

    gene_order_db, gene_coord_db = load_gff_data(args.pharokka_dir, locus_to_group_map, genome_ids)
    df_gpa = pd.read_csv(args.panaroo_file, low_memory=False)
    gene_annotation_map = pd.Series(df_gpa['Annotation'].values, index=df_gpa['Gene']).to_dict()
    protein_db = load_protein_sequences(args.faa_dir, locus_to_group_map)
    genome_seqs = load_genome_sequences(args.fasta_dir, genome_ids)
    passed_genomes, qc_report = run_qc(
        genome_ids, args.fasta_dir, gene_order_db, gene_coord_db, 
        max_contigs=getattr(args, 'max_contigs', 1),
        orientation_threshold=getattr(args, 'orientation_consistency_threshold', 0.9)
    )

    # Step 3: Anchor Discovery
    params_da = {'tolerance': args.tolerance, 'coverage_threshold': args.coverage_threshold, 'min_length': args.min_length}
    anchors_basic = discover_anchors(gene_order_db, stratified_genes['strict_core'], passed_genomes, **params_da)
    
    # --- THIS IS THE MODIFIED PART ---
    # Check if any anchors were found. If not, we stop further analysis but create a valid output file.
    if not anchors_basic:
        print("\n" + "="*80)
        print("[CRITICAL WARNING] No anchor modules were found that meet the specified criteria.")
        print("This means no stable genomic backbone could be identified for this dataset.")
        print("Downstream analyses (slot_analysis, etc.) will be skipped as they depend on anchors.")
        print("\nPossible reasons:")
        print("  1. The pangenome has very few 'strict core' genes.")
        print("  2. The genomic structure is highly variable across the cohort.")
        print("  3. The '--coverage-threshold' or '--min-length' parameters are too strict.")
        print("\nSuggestions:")
        print("  - Try re-running with a lower '--coverage-threshold' (e.g., 0.8) or a lower '--min-length' (e.g., 2).")
        print("="*80 + "\n")
        
        # Create an empty but valid output structure
        final_output = {
            "metadata": {
                "analysis_parameters": vars(args), 
                "qc_summary": qc_report,
                "pangenome_stratification": stratified_genes,
                "status_message": "No anchors found. Analysis halted."
            },
            "anchor_definitions": [],
            "macro_architectures": {}
        }
    else:
        # If anchors were found, proceed with the normal workflow
        anchors_characterized = characterize_anchors(anchors_basic, protein_db, gene_order_db, gene_coord_db, genome_seqs, passed_genomes, intermediate_dir)
        anchors_functional = annotate_anchor_functions(anchors_characterized, gene_annotation_map)
        macro_report = analyze_macro_architecture(anchors_functional, gene_order_db, passed_genomes)
        
        final_output = {
            "metadata": {
                "analysis_parameters": vars(args), 
                "qc_summary": qc_report,
                "pangenome_stratification": stratified_genes
            },
            "anchor_definitions": anchors_functional,
            "macro_architectures": macro_report
        }
    # --- END OF MODIFICATION ---

    # Save intermediate files regardless, as they can be useful for diagnostics
    print("\n[*] Saving intermediate databases for downstream modules or diagnostics...")
    save_json(stratified_genes, os.path.join(intermediate_dir, "pangenome_stratification.json"))
    save_json(locus_to_group_map, os.path.join(intermediate_dir, "locus_to_group_map.json"))
    save_json(gene_order_db, os.path.join(intermediate_dir, "gene_order_database.json"))
    save_json(gene_coord_db, os.path.join(intermediate_dir, "gene_coordinate_database.json"))
    save_json(gene_annotation_map, os.path.join(intermediate_dir, "gene_annotation_map.json"))
    print("[*] Intermediate files saved successfully.")

    # Save the final (possibly empty) architecture definition
    final_output_path = os.path.join(output_dir, "architecture_definition.json")
    save_json(final_output, final_output_path)
    
    print(f"\n====== 'define_architecture' Module Finished Successfully! ======")
    print(f"[*] Final output saved to: {final_output_path}")
    return True

define_architecture/data_loader.py

In [ ]:
import pandas as pd
import os
import json
from collections import defaultdict
from tqdm import tqdm
from Bio import SeqIO

def load_panaroo_data(panaroo_file, strict_core_threshold=1.0, soft_core_threshold=0.98):
    """
    Parses the Panaroo gene_presence_absence_roary.csv file.
    """
    print("--- Task 1.1: Loading Panaroo Data ---")
    try:
        df_gpa = pd.read_csv(panaroo_file, low_memory=False)
    except FileNotFoundError:
        print(f"[ERROR] Panaroo file not found at: {panaroo_file}"); exit(1)

    genome_ids = df_gpa.columns[14:].tolist()
    num_genomes = len(genome_ids)
    print(f"Found {num_genomes} genomes in the Panaroo matrix.")

    locus_to_group_map = {}
    print("Building locus_tag -> group_id map...")
    for _, row in tqdm(df_gpa.iterrows(), total=len(df_gpa), desc="  Parsing rows"):
        group_id = row['Gene']
        for genome_id in genome_ids:
            if pd.notna(row[genome_id]):
                for tag in str(row[genome_id]).split(';'):
                    if tag: locus_to_group_map[tag.strip()] = group_id
    
    print("Stratifying genes into core/accessory categories...")
    strict_core, soft_core, accessory = [], [], []
    df_gpa['prevalence'] = df_gpa['No. isolates'] / num_genomes
    for _, row in df_gpa.iterrows():
        if row['prevalence'] >= strict_core_threshold: strict_core.append(row['Gene'])
        elif row['prevalence'] >= soft_core_threshold: soft_core.append(row['Gene'])
        else: accessory.append(row['Gene'])
            
    stratified_genes = {'strict_core': strict_core, 'soft_core': soft_core, 'accessory': accessory}
    print("Panaroo data loading complete.")
    return genome_ids, stratified_genes, locus_to_group_map


def load_gff_data(gff_dir, locus_to_group_map, genome_ids):
    """
    Parses all GFF files from a directory to build gene order and coordinate databases.
    """
    print("\n--- Task 1.2: Loading GFF Data (Robust Version) ---")
    gene_order_db, gene_coord_db = {}, defaultdict(list)
    available_subdirs = {d for d in os.listdir(gff_dir) if os.path.isdir(os.path.join(gff_dir, d))}
    
    for genome_id in tqdm(genome_ids, desc="Parsing GFFs"):
        if genome_id not in available_subdirs:
            print(f"  [Warning] No subdirectory named '{genome_id}' found in {gff_dir}. Skipping."); gene_order_db[genome_id], gene_coord_db[genome_id] = [], []; continue

        gff_path = os.path.join(gff_dir, genome_id, f"{genome_id}.gff")
        if not os.path.exists(gff_path):
            print(f"  [Warning] GFF file not found at {gff_path}. Skipping."); gene_order_db[genome_id], gene_coord_db[genome_id] = [], []; continue
            
        genes_with_pos = []
        with open(gff_path, 'r') as f:
            for line in f:
                if line.startswith('#') or '\tCDS\t' not in line: continue
                parts = line.strip().split('\t')
                attributes = {k: v for k, v in [attr.split('=') for attr in parts[8].split(';') if '=' in attr]}
                locus_tag = attributes.get('ID')
                if locus_tag and locus_tag in locus_to_group_map:
                    group_id = locus_to_group_map[locus_tag]
                    gene_info = { 'group_id': group_id, 'contig': parts[0], 'start': int(parts[3]), 'end': int(parts[4]), 'strand': parts[6], 'locus_tag': locus_tag }
                    genes_with_pos.append(gene_info)
                    
        gene_coord_db[genome_id].extend(genes_with_pos)
        genes_with_pos.sort(key=lambda x: x['start'])
        gene_order_db[genome_id] = [g['group_id'] for g in genes_with_pos]
        
    print("GFF data loading complete.")
    return gene_order_db, gene_coord_db


def load_protein_sequences(protein_faa_dir, locus_to_group_map):
    """
    Loads all protein sequences from phanotate.faa files for ASI calculation.
    """
    print("\n--- Task: Loading Protein Sequences for ASI ---")
    protein_db = defaultdict(dict)
    genome_dirs = [d for d in os.listdir(protein_faa_dir) if os.path.isdir(os.path.join(protein_faa_dir, d))]
    
    for genome_id in tqdm(genome_dirs, desc="Loading FAAs"):
        faa_path = os.path.join(protein_faa_dir, genome_id, "phanotate.faa")
        if not os.path.exists(faa_path): continue
            
        for record in SeqIO.parse(faa_path, "fasta"):
            locus_tag = record.id
            if locus_tag in locus_to_group_map:
                group_id = locus_to_group_map[locus_tag]
                protein_db[group_id][genome_id] = str(record.seq)
                
    print(f"Loaded protein sequences for {len(protein_db)} gene families.")
    return protein_db


def load_genome_sequences(fasta_dir, genome_ids):
    """
    Loads all genome sequences from a directory for boundary signal analysis.
    """
    print("\n--- Task: Loading Genome Sequences for Boundary Analysis ---")
    genome_seqs = {}
    
    for genome_id in tqdm(genome_ids, desc="Loading FASTAs"):
        fasta_path = None
        for ext in ['.fasta', '.fa', '.fna']:
            potential_path = os.path.join(fasta_dir, f"{genome_id}{ext}")
            if os.path.exists(potential_path):
                fasta_path = potential_path
                break
        
        if fasta_path is None: continue
            
        try:
            genome_seqs[genome_id] = SeqIO.read(fasta_path, "fasta")
        except Exception as e:
            print(f"  [Error] Could not parse FASTA for {genome_id}: {e}")

    print(f"Loaded {len(genome_seqs)} genome sequences.")
    return genome_seqs

define_architecture/quality_control.py

In [ ]:
# /GenoSynax_test/genosyntax/define_architecture/quality_control.py

import os
from collections import Counter
from Bio import SeqIO

def analyze_fasta_for_qc(fasta_path):
    """
    A lightweight and robust function to analyze a FASTA file for QC purposes.
    VERSION 2 (ROBUST): Instead of relying on Bio.SeqIO.parse(), this version
    reads the file content directly and counts '>' occurrences. This is more
    resilient to non-standard line endings and other formatting issues that
    can trick the parser into seeing multiple contigs where there is only one.
    """
    if not os.path.exists(fasta_path):
        return {'num_contigs': -1, 'error': 'FASTA file not found'}
    
    try:
        contig_count = 0
        with open(fasta_path, 'r', errors='ignore') as f:
            # Read the file line by line to count headers
            for line in f:
                if line.startswith('>'):
                    contig_count += 1
        
        # If the file is empty or contains no headers, it's an issue.
        if contig_count == 0:
            return {'num_contigs': 0, 'error': 'No FASTA headers found in file'}
            
        return {'num_contigs': contig_count}

    except Exception as e:
        # This will catch file reading errors, encoding errors, etc.
        return {'num_contigs': -1, 'error': f'File reading failed: {e}'}


def run_qc(genome_ids, fasta_dir, gene_order_db, gene_coord_db, max_contigs=1, orientation_threshold=0.9):
    """
    Performs QC checks.
    This function now uses the robust analyze_fasta_for_qc v2.
    """
    print("\n--- Task 2: Running Quality Control (using robust contig counting) ---")
    
    total_genomes_input = genome_ids
    excluded_genomes = {}
    
    # --- QC Check 1: Contig Count from SOURCE FASTA files ---
    print(f"QC 1: Checking contig counts from FASTA files in '{fasta_dir}' (threshold: max_contigs <= {max_contigs})...")
    genomes_after_contig_qc = []
    fasta_contig_counts = {}
    
    for genome_id in total_genomes_input:
        fasta_path = None
        # Robustly find the FASTA file with common extensions
        for ext in ['.fasta', '.fa', '.fna']:
            potential_path = os.path.join(fasta_dir, f"{genome_id}{ext}")
            if os.path.exists(potential_path):
                fasta_path = potential_path
                break
        
        # This now calls our new, robust function
        qc_result = analyze_fasta_for_qc(fasta_path)
        num_contigs = qc_result.get('num_contigs')
        fasta_contig_counts[genome_id] = num_contigs
        
        if num_contigs in [-1, 0]: # Handle file not found or empty/no-header files
            reason = qc_result.get('error', 'Unknown file error')
            excluded_genomes[genome_id] = f"QC Error: {reason}"
            continue
            
        if not gene_coord_db.get(genome_id):
            excluded_genomes[genome_id] = "Data Loading Failed (GFF not found or empty)"
            continue
            
        if num_contigs > max_contigs:
            excluded_genomes[genome_id] = f"Fragmented Assembly ({num_contigs} contigs)"
        else:
            genomes_after_contig_qc.append(genome_id)

    print(f"  {len(genomes_after_contig_qc)}/{len(total_genomes_input)} genomes passed all filtering criteria.")

    # --- QC Check 2: Orientation Consistency ---
    print(f"QC 2: Checking orientation consistency (threshold: >{orientation_threshold*100}%)...")
    if not genomes_after_contig_qc:
        orientation_summary = {
            'is_consistently_oriented': False,
            'orientation_gene': None,
            'confidence': 0.0,
            'details': "No high-quality genomes available for analysis."
        }
    else:
        # Ensure we only check genomes that have a gene order available
        valid_genomes_for_orient = [gid for gid in genomes_after_contig_qc if gene_order_db.get(gid)]
        
        if not valid_genomes_for_orient:
             orientation_summary = {
                'is_consistently_oriented': False,
                'orientation_gene': None,
                'confidence': 0.0,
                'details': "High-quality genomes have empty gene orders."
            }
        else:
            start_genes = [gene_order_db[gid][0] for gid in valid_genomes_for_orient]
            start_gene_counts = Counter(start_genes)
            
            dominant_start_gene, count = start_gene_counts.most_common(1)[0]
            confidence = count / len(valid_genomes_for_orient)
            is_consistent = confidence >= orientation_threshold
            
            orientation_summary = {
                'is_consistently_oriented': is_consistent,
                'orientation_gene': dominant_start_gene,
                'confidence': round(confidence, 4),
                'details': f"Dominant start gene '{dominant_start_gene}' found in {count}/{len(valid_genomes_for_orient)} ({confidence:.2%}) of high-quality genomes."
            }
        print(f"  Orientation check complete. Consistent: {orientation_summary['is_consistently_oriented']}.")

    # --- Final Report Generation ---
    qc_report = {
        'parameters': { 
            'max_contigs': max_contigs, 
            'orientation_consistency_threshold': orientation_threshold 
        },
        'summary': { 
            'total_genomes_input': len(total_genomes_input), 
            'genomes_passed_qc': len(genomes_after_contig_qc), 
            'genomes_failed_qc': len(excluded_genomes) 
        },
        'failed_genomes': excluded_genomes,
        'fasta_contig_counts': fasta_contig_counts,
        'orientation_analysis': orientation_summary,
        'passed_genomes_list': sorted(genomes_after_contig_qc)
    }
    
    print("Quality control finished.")
    return genomes_after_contig_qc, qc_report

define_architecture/anchor_discovery.py

In [ ]:
import os
from tqdm import tqdm
import numpy as np
from scipy.stats import spearmanr
from itertools import combinations
from genosyntax.utils.file_ops import calculate_asi_for_gene, calculate_gc_content


def discover_anchors(gene_order_db, strict_core_genes, passed_genomes, 
                     tolerance, coverage_threshold, min_length):
    print("\n--- Task 3: Discovering Anchor Modules ---")
    if not passed_genomes: return []
    reference_genome_id = passed_genomes[0]
    reference_order = gene_order_db[reference_genome_id]
    candidates, current_block, accessory_count = [], [], 0
    for gene in reference_order:
        if gene in strict_core_genes:
            current_block.append(gene); accessory_count = 0
        else:
            accessory_count += 1
            if accessory_count > tolerance:
                if len(current_block) >= min_length: candidates.append(tuple(current_block))
                current_block = []
    if len(current_block) >= min_length: candidates.append(tuple(current_block))
    unique_candidates = sorted(list(set(candidates)), key=len, reverse=True)
    print(f"Identified {len(unique_candidates)} unique candidate skeletons from reference '{reference_genome_id}'.")
    print("Verifying candidates across all genomes...")
    skeleton_to_ref_pos = {sk: reference_order.index(sk[0]) for sk in unique_candidates}
    valid_anchors, num_genomes, min_genomes_required = [], len(passed_genomes), len(passed_genomes) * coverage_threshold
    for skeleton in tqdm(unique_candidates, desc="  Verifying skeletons"):
        count = sum(1 for genome_id in passed_genomes if _verify_skeleton_in_genome(skeleton, gene_order_db[genome_id], strict_core_genes, tolerance))
        if count >= min_genomes_required:
            valid_anchors.append({"skeleton": list(skeleton), "core_gene_count": len(skeleton), "coverage_count": count, "coverage_fraction": round(count / num_genomes, 4), "reference_position": skeleton_to_ref_pos[skeleton]})
    print(f"Found {len(valid_anchors)} anchors meeting the {coverage_threshold:.0%} coverage threshold.")
    return valid_anchors

def _verify_skeleton_in_genome(skeleton, genome_order, core_genes, tolerance):
    skel_ptr, skel_len, order_ptr, order_len = 0, len(skeleton), 0, len(genome_order)
    while order_ptr < order_len:
        try: start_pos = genome_order.index(skeleton[0], order_ptr)
        except ValueError: return False
        skel_ptr, genome_ptr, acc_count = 1, start_pos + 1, 0
        while skel_ptr < skel_len and genome_ptr < order_len:
            if genome_order[genome_ptr] == skeleton[skel_ptr]: skel_ptr += 1; acc_count = 0
            elif genome_order[genome_ptr] not in core_genes:
                acc_count += 1
                if acc_count > tolerance: break
            else: break;
            genome_ptr += 1
        if skel_ptr == skel_len: return True
        order_ptr = start_pos + 1
    return False


def characterize_anchors(anchors, protein_db, gene_order_db, gene_coord_db, 
                       genome_seqs, # <-- THIS WILL NOW RECEIVE THE CORRECT SEQUENCES
                       passed_genomes, temp_dir, boundary_window=200):
    
    # The logic of this function was already correct, it just needed the correct `genome_seqs` input.
    # No changes are needed inside this function itself.
    print("\n--- Task 4: Characterizing Anchor Modules (Full Version) ---")
    print("  Pre-calculating global background GC content...")
    global_gc = [calculate_gc_content(str(genome_seqs[gid].seq)) for gid in passed_genomes if gid in genome_seqs and str(genome_seqs[gid].seq)]
    background_gc_mean, background_gc_std = np.mean(global_gc) if global_gc else 0.0, np.std(global_gc) if global_gc else 1.0
    anchor_presence_map = {}
    anchors.sort(key=lambda x: x['reference_position'])
    all_core_genes = {gene for sk in anchors for gene in sk['skeleton']}
    for i, anchor in enumerate(anchors):
        anchor_id = f"Anchor_{i+1}"; anchor['anchor_id'] = anchor_id
        anchor_presence_map[anchor_id] = [gid for gid in passed_genomes if _verify_skeleton_in_genome(tuple(anchor['skeleton']), gene_order_db[gid], all_core_genes, 2)]
    for anchor in tqdm(anchors, desc="  Characterizing anchors"):
        skeleton, present_genomes = anchor['skeleton'], anchor_presence_map[anchor['anchor_id']]
        all_gene_asis = [asi for core_gene in skeleton if (asi := calculate_asi_for_gene([s for gid in present_genomes if (s := protein_db.get(core_gene, {}).get(gid))], temp_dir)) is not None]
        anchor['asi'] = round(np.mean(all_gene_asis), 4) if all_gene_asis else 0.0
        rank_data = [ranks for gid in present_genomes if len(ranks := [gene_order_db[gid].index(g) for g in skeleton if g in gene_order_db[gid]]) == len(skeleton)]
        if len(rank_data) < 2: anchor['aoi'] = 1.0
        else:
            rank_matrix = np.array(rank_data)
            correlations = [spearmanr(rank_matrix[:, i], rank_matrix[:, j])[0] for i, j in combinations(range(len(skeleton)), 2)]
            anchor['aoi'] = round(np.mean([c for c in correlations if not np.isnan(c)]), 4) if correlations else 1.0
        boundary_gc_zscores = []
        first_gene, last_gene = skeleton[0], skeleton[-1]
        for genome_id in present_genomes:
            if genome_id not in genome_seqs or genome_id not in gene_coord_db: continue
            genome_sequence, coords = str(genome_seqs[genome_id].seq), gene_coord_db[genome_id]
            first_coord, last_coord = next((g for g in coords if g['group_id'] == first_gene), None), next((g for g in coords if g['group_id'] == last_gene), None)
            if not first_coord or not last_coord: continue
            up_end, up_start = first_coord['start'] - 1, max(0, first_coord['start'] - 1 - boundary_window)
            if up_end > up_start: boundary_gc_zscores.append((calculate_gc_content(genome_sequence[up_start:up_end]) - background_gc_mean) / background_gc_std if background_gc_std > 0 else 0)
            down_start, down_end = last_coord['end'], min(len(genome_sequence), last_coord['end'] + boundary_window)
            if down_end > down_start: boundary_gc_zscores.append((calculate_gc_content(genome_sequence[down_start:down_end]) - background_gc_mean) / background_gc_std if background_gc_std > 0 else 0)
        anchor['boundary_signals'] = {"window_size_bp": boundary_window, "avg_gc_enrichment_zscore": round(np.mean(boundary_gc_zscores), 4) if boundary_gc_zscores else 0.0}
    return anchors

define_architecture/macro_analysis.py

In [ ]:
# /GenoSynax_test/genosyntax/define_architecture/macro_analysis.py

from collections import Counter, defaultdict
from scipy.stats import entropy

# --- Robust Import ---
# It is critical to use the same anchor verification logic as the rest of the pipeline.
try:
    # This works when running as part of the genosyntax package.
    from .anchor_discovery import _verify_skeleton_in_genome
except ImportError:
    # This is a fallback for standalone testing of the script.
    from anchor_discovery import _verify_skeleton_in_genome

# --- Helper Functions ---

def _find_anchor_start_pos(skeleton, genome_order):
    """
    Helper function to find the starting position of an anchor's skeleton.
    Uses the position of the first gene in the skeleton as a proxy for sorting.
    Returns -1 if the first gene is not found, ensuring stability.
    """
    try:
        return genome_order.index(skeleton[0])
    except ValueError:
        return -1

def is_subsequence(child_perm, parent_perm):
    """
    Checks if one tuple/list ('child_perm') is an ordered subsequence of another ('parent_perm').
    This is the core of the flexible grouping logic.
    e.g., is_subsequence(('A', 'C'), ('A', 'B', 'C')) -> True
    e.g., is_subsequence(('C', 'A'), ('A', 'B', 'C')) -> False
    """
    it = iter(parent_perm)
    return all(element in it for element in child_perm)

# --- Main Analysis Function ---

def analyze_macro_architecture(anchors, gene_order_db, passed_genomes, tolerance=2):
    """
    Analyzes the permutation order of anchor modules across all high-quality genomes.

    VERSION 3 (ROBUST - ORDERED SUBSET):
    This version implements a robust, two-stage process to correctly classify genomic architectures:
    1.  It first uses a strict verification function (`_verify_skeleton_in_genome`) to accurately
        identify the presence of structurally complete anchors in each genome.
    2.  It then groups genomes using an "Ordered Subset" (subsequence) matching algorithm.
        This ensures that genomes with significant anchor loss (e.g., containing only 2 of 10 anchors)
        are correctly classified as part of a major architecture, provided their remaining anchors
        maintain the correct relative order. This prevents incorrect fragmentation into multiple
        "chaotic" architectures.

    Args:
        anchors (list): List of anchor definition dictionaries from previous steps.
        gene_order_db (dict): Maps genome_id to its ordered list of gene group IDs.
        passed_genomes (list): List of genome IDs that passed quality control.
        tolerance (int): The number of non-core genes tolerated within an anchor skeleton.

    Returns:
        dict: A report summarizing the macro-architectural landscape, including major/minor
              architectures and global permutation entropy.
    """
    print("\n--- Task 5: Analyzing Macro-Architecture (Robust Ordered Subset Version) ---")

    if not anchors:
        print("  [Warning] No anchor definitions provided. Skipping macro-analysis.")
        return {}

    # --- STEP 1: Robustly locate all valid anchors and their order in each genome ---
    print("  Step 1: Accurately locating structurally complete anchors in each genome...")
    genome_permutations = {}
    anchor_map = {a['anchor_id']: a for a in anchors}
    all_core_genes = {gene for anchor in anchors for gene in anchor['skeleton']}

    for genome_id in passed_genomes:
        gene_order = gene_order_db.get(genome_id)
        if not gene_order: continue

        located_anchors = []
        for anchor_id, anchor_info in anchor_map.items():
            skeleton = tuple(anchor_info['skeleton'])
            
            # Use the robust verification function to ensure the entire anchor is present.
            if _verify_skeleton_in_genome(skeleton, gene_order, all_core_genes, tolerance):
                start_pos = _find_anchor_start_pos(skeleton, gene_order)
                if start_pos != -1:
                    located_anchors.append({'pos': start_pos, 'id': anchor_id})
        
        if located_anchors:
            # Sort the found anchors by their physical position in the genome.
            located_anchors.sort(key=lambda x: x['pos'])
            genome_permutations[genome_id] = tuple(a['id'] for a in located_anchors)

    if not genome_permutations:
        print("  [Warning] No valid anchor permutations could be determined for any genome.")
        return {}

    # --- STEP 2: Group architectures based on the "Ordered Subset" principle ---
    print("  Step 2: Grouping architectures via ordered subset matching...")
    
    unique_perms = sorted(Counter(genome_permutations.values()).keys(), key=len, reverse=True)
    
    # This map will store the "ultimate parent" for every observed permutation.
    parent_map = {p: p for p in unique_perms} # Initially, each permutation is its own parent.

    # Two-pass approach to resolve the full hierarchy of parent-child relationships.
    for p_parent in unique_perms:
        for p_child in unique_perms:
            if len(p_child) >= len(p_parent): continue # A child cannot be longer than its parent.
            if p_child == p_parent: continue

            # If the child is a subsequence of the parent, map the child to the parent.
            if is_subsequence(p_child, parent_map[p_parent]):
                 parent_map[p_child] = parent_map[p_parent]

    # Second pass for path compression: ensure all mappings point to the ultimate root parent.
    for perm in unique_perms:
        parent = parent_map[perm]
        while parent != parent_map[parent]:
            parent = parent_map[parent]
        parent_map[perm] = parent
        
    # --- STEP 3: Assemble the final report from the grouped architectural families ---
    print(f"  Step 3: Assembling final report for {len(set(parent_map.values()))} distinct architectural families...")
    
    architecture_families = defaultdict(list)
    for genome_id, perm in genome_permutations.items():
        final_parent_perm = parent_map[perm]
        architecture_families[final_parent_perm].append(genome_id)

    final_groups = []
    for parent_perm, genomes in architecture_families.items():
        final_groups.append({
            "permutation": list(parent_perm),
            "supporting_genomes": genomes,
        })

    # Sort final architecture groups by the number of supporting genomes.
    sorted_groups = sorted(final_groups, key=lambda x: len(x['supporting_genomes']), reverse=True)
    
    # --- STEP 4: Calculate global entropy and finalize the report ---
    major_architecture = sorted_groups[0] if sorted_groups else {}
    minor_architectures = sorted_groups[1:] if len(sorted_groups) > 1 else []

    total_genomes_with_arch = len(genome_permutations)
    frequencies = [len(g['supporting_genomes']) / total_genomes_with_arch for g in sorted_groups]
    global_entropy = entropy(frequencies, base=2) if frequencies else 0.0

    # Add frequency information to the report.
    if major_architecture:
        major_architecture['frequency'] = round(len(major_architecture['supporting_genomes']) / total_genomes_with_arch, 4)
    for group in minor_architectures:
        group['frequency'] = round(len(group['supporting_genomes']) / total_genomes_with_arch, 4)

    macro_report = {
        "global_permutation_entropy": round(global_entropy, 4),
        "total_architectures_found": len(sorted_groups),
        "major_architecture": major_architecture,
        "minor_architectures": minor_architectures
    }
    
    print("Macro-architecture analysis complete.")
    return macro_report

define_architecture/functional_annotator.py

In [ ]:
from collections import Counter

# Define a dictionary of keywords for functional super-classes.
# This is the "knowledge base" of our annotation engine.
# It should be easily configurable.
FUNCTIONAL_MAP = {
    "Virion Morphogenesis & Infection": [
        'tail', 'baseplate', 'capsid', 'head', 'portal', 'terminase', 
        'lysis', 'holin', 'endolysin', 'sheath', 'fiber'
    ],
    "Host Takeover & Genome Replication": [
        'dna polymerase', 'helicase', 'primase', 'ligase', 'replication',
        'rna polymerase', 'transcription', 'sigma factor'
    ],
    "Genome Maintenance & Host Interaction": [
        'recombination', 'exonuclease', 'endonuclease', 'repair', 
        'nucleotide', 'synthesis', 'kinase', 'synthase', 'reductase',
        'metabolism', 'folate'
    ],
    "Unknown & Other": [
        'hypothetical', 'unknown', 'uncharacterized'
    ]
}

def annotate_anchor_functions(anchors, gene_annotation_map):
    """
    Assigns a high-level functional category to each anchor module based on the
    functions of its member genes.

    Args:
        anchors (list): The list of characterized anchor definition dictionaries.
        gene_annotation_map (dict): A map from group_id to functional description.

    Returns:
        list: The same list of anchors, with 'functional_class' and 
              'function_profile' fields added to each anchor dictionary.
    """
    print("\n--- Task 6: Annotating Anchor Functions ---")
    
    for anchor in anchors:
        skeleton = anchor['skeleton']
        function_counts = Counter()

        # Get annotations for all genes in the anchor's skeleton
        gene_functions = [gene_annotation_map.get(gene_id, 'unknown') for gene_id in skeleton]
        
        # Classify each gene's function
        for func_desc in gene_functions:
            assigned_class = "Unknown & Other" # Default
            for category, keywords in FUNCTIONAL_MAP.items():
                # Use lower case for robust matching
                if any(keyword in func_desc.lower() for keyword in keywords):
                    assigned_class = category
                    break
            function_counts[assigned_class] += 1
            
        # Determine the dominant functional class for the anchor
        # We prioritize non-unknown functions
        non_unknown_counts = {k: v for k, v in function_counts.items() if k != "Unknown & Other"}
        
        if non_unknown_counts:
            # The class is the one with the most member genes
            dominant_class = max(non_unknown_counts, key=non_unknown_counts.get)
        else:
            dominant_class = "Unknown & Other"
            
        anchor['functional_class'] = dominant_class
        anchor['function_profile'] = dict(function_counts)

    print("Functional annotation complete.")
    return anchors